In [80]:
import warnings
warnings.filterwarnings('ignore')

from sigpyproc.readers import FilReader
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
from matplotlib import dates
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from datetime import datetime, timedelta
import astropy.units as u
from astropy.time import Time
from astropy.visualization import ImageNormalize, PercentileInterval
from scipy.optimize import fsolve
from tqdm import tqdm

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

path = '/home/mnedal/data/I-LOFAR'
data_dir = '/home/mnedal/data'

In [3]:
import glob

files = glob.glob(f'{path}/*.fil')
for i, file in enumerate(files):
    print(i, file)

0 /home/mnedal/data/I-LOFAR/realta_ilofar_stokesI_20240514T1635_masked.fil
1 /home/mnedal/data/I-LOFAR/Sun357_2025-03-26_typeII_2.fil
2 /home/mnedal/data/I-LOFAR/Sun357_20240514_stokesI.fil
3 /home/mnedal/data/I-LOFAR/realta_ilofar_stokesI_20240514T1635.fil
4 /home/mnedal/data/I-LOFAR/Sun357_20240514_stokesV.fil
5 /home/mnedal/data/I-LOFAR/Sun357_2025-03-26T08:31:00_StokesI_dyspec.fil
6 /home/mnedal/data/I-LOFAR/Sun357_2025-03-25_typeII.fil
7 /home/mnedal/data/I-LOFAR/Sun357_2025-03-26_typeII_1.fil


In [4]:
i = 2
filename = files[i]
a = FilReader(filename) # header
header = a.header.to_dict()
display(header)

{'filename': '/home/mnedal/data/I-LOFAR/Sun357_20240514_stokesI.fil',
 'data_type': 'filterbank',
 'nchans': 488,
 'foff': -0.1953125,
 'fch1': 200.0,
 'nbits': 32,
 'tsamp': 0.00131072,
 'tstart': 60444.418055551,
 'nsamples': 24902343,
 'nifs': 1,
 'coord': <SkyCoord (ICRS): (ra, dec) in deg
     (0., 0.)>,
 'azimuth': <Angle 297.27495739 deg>,
 'zenith': <Angle 52.16498088 deg>,
 'telescope': 'MeerKAT',
 'backend': 'FAKE',
 'source': 'Sun357',
 'frame': 'topocentric',
 'ibeam': 0,
 'nbeams': 0,
 'dm': 0,
 'period': 0,
 'accel': 0,
 'signed': False,
 'rawdatafile': 'unknown',
 'stream_info': {'entries': [{'filename': '/home/mnedal/data/I-LOFAR/Sun357_20240514_stokesI.fil',
    'hdrlen': 347,
    'datalen': 48609373536,
    'nsamples': 24902343,
    'tstart': 60444.418055551,
    'tsamp': 0.00131072}]},
 'basename': 'Sun357_20240514_stokesI',
 'extension': '.fil',
 'telescope_id': 64,
 'machine_id': 0,
 'bandwidth': 95.3125,
 'ftop': 200.09765625,
 'fbottom': 104.78515625,
 'fcenter':

In [5]:
# Extract stokes param and datetime str from the file path
import os

fname = os.path.basename(filename)
parts = fname.split('_')

stokes = None
date_or_datetime = None

for p in parts:
    # stokes parameter
    if p.startswith('stokes'):
        stokes = p

    # full datetime: YYYYMMDDThhmm
    elif (
        'T' in p 
        and len(p) == 13 
        and p[:8].isdigit() 
        and p[9:].isdigit()
    ):
        date_or_datetime = p

    # pure date: YYYYMMDD
    elif len(p) == 8 and p.isdigit():
        date_or_datetime = p

print(stokes, date_or_datetime)

stokesI.fil 20240514


In [6]:
tstart_obs_str = Time(a.header.tstart, format='mjd').iso
n_samples      = a.header.nsamples
print(f'Start obs time: {tstart_obs_str}\nNo. of samples: {n_samples}')

Start obs time: 2024-05-14 10:02:00.000
No. of samples: 24902343


In [7]:
data = a.read_block(start=0, nsamps=n_samples)
print(data.shape) # (freqs, times)

(488, 24902343)


In [8]:
# making time axis
tstart = Time(data.header.tstart, format='mjd')                    # tstart.iso will tell the time in format yyyy-mm-dd hh:mm:ss
tarray = tstart + (np.arange(data.shape[1])*data.header.tsamp*u.s) # making the time array for realta time resolution
print(f'Length of array: {len(tarray)}\nStart time: {tarray[0].iso}\nEnd time: {tarray[-1].iso}')

Length of array: 24902343
Start time: 2024-05-14 10:02:00.000
End time: 2024-05-14 19:05:59.997


In [9]:
dt = datetime.strptime(tarray[1].iso, '%Y-%m-%d %H:%M:%S.%f') - datetime.strptime(tarray[0].iso, '%Y-%m-%d %H:%M:%S.%f')
print('Time cadence:', dt.total_seconds()*1000, 'ms')

Time cadence: 1.0 ms


In [11]:
# Converting the array to datetime object --> expand the for-loop and use tqdm since this step takes a long time!
# Tarray = [datetime.strptime(t.iso, '%Y-%m-%d %H:%M:%S.%f') for t in tarray]
if not os.path.exists(f'{path}/timesArray_realta_{date_or_datetime}.npy'):
    Tarray = []
    with tqdm(total=len(tarray), desc='Converting time array to datetime object') as pbar:
        for t in tarray:
            tmp_var = datetime.strptime(t.iso, '%Y-%m-%d %H:%M:%S.%f')
            Tarray.append(tmp_var)
            pbar.update(1)
    
    np.save(f'{path}/timesArray_realta_{date_or_datetime}.npy', Tarray)
    print(Tarray[0], Tarray[-1], sep='\n')
else:
    Tarray = np.load(f'{path}/timesArray_realta_{date_or_datetime}.npy', allow_pickle=True)

In [12]:
type(Tarray)

numpy.ndarray

In [13]:
# export the frequency axis
freqs = data.header.chan_freqs
print(f'Start freq: {freqs[0]:.2f}\nEnd freq: {freqs[-1]:.2f}')

Start freq: 200.00
End freq: 104.88


In [14]:
def freq_axis(freqs):
    """
    Introduce gaps in the frequency axis of I-LOFAR REALTA data.
    """
    gap1 = np.flipud(freqs[288]+(np.arange(59)*0.390625))
    gap2 = np.flipud(freqs[88]+(np.arange(57)*0.390625))
    ax_shape = 59+57-1
    new_freq = np.zeros(ax_shape+freqs.shape[0])
    
    new_freq[0:88]    = freqs[0:88]
    new_freq[88:145]  = gap2[:57]
    new_freq[145:345] = freqs[88:288]
    new_freq[345:404] = gap1[:59]
    new_freq[404:]    = freqs[289:]
    
    return new_freq

In [15]:
new_freq = freq_axis(freqs)

data = np.log10(data)
data[np.where(np.isinf(data)==True)] = 0.0

data2          = np.empty((new_freq.shape[0], data.shape[1]))    
data2[:]       = np.NaN
data2[0:88]    = data[0:88]
data2[145:345] = data[88:288]
data2[404:]    = data[289:]

In [16]:
freq_mode3 = np.linspace(10, 90, 199)
freq_mode5 = np.linspace(110, 190, 200)
freq_mode7 = np.linspace(210, 270, 88)

df_mode3 = pd.DataFrame(data=data2[404:].T, columns=freq_mode3[::-1])
df_mode5 = pd.DataFrame(data=data2[145:345].T, columns=freq_mode5[::-1])
df_mode7 = pd.DataFrame(data=data2[:88].T, columns=freq_mode7[::-1])

In [17]:
print(df_mode3.shape, df_mode5.shape, df_mode7.shape, sep='\n')

(24902343, 199)
(24902343, 200)
(24902343, 88)


In [18]:
# Ensure the 'time' column is in datetime format
df_mode3.index = pd.to_datetime(Tarray)
df_mode5.index = pd.to_datetime(Tarray)
df_mode7.index = pd.to_datetime(Tarray)

In [19]:
# Save the dataframes as a pickle files
if not os.path.exists(f'{path}/df_mode3_realta_{date_or_datetime}_stokes{stokes}.pkl'):
    df_mode3.to_pickle(f'{path}/df_mode3_realta_{date_or_datetime}_stokes{stokes}.pkl')

if not os.path.exists(f'{path}/df_mode5_realta_{date_or_datetime}_stokes{stokes}.pkl'):
    df_mode5.to_pickle(f'{path}/df_mode5_realta_{date_or_datetime}_stokes{stokes}.pkl')

if not os.path.exists(f'{path}/df_mode7_realta_{date_or_datetime}_stokes{stokes}.pkl'):
    df_mode7.to_pickle(f'{path}/df_mode7_realta_{date_or_datetime}_stokes{stokes}.pkl')

---

## Load the dataframes from the pickle files

In [20]:
df_mode3 = pd.read_pickle(f'{path}/df_mode3_realta_{date_or_datetime}_stokes{stokes}.pkl')
df_mode5 = pd.read_pickle(f'{path}/df_mode5_realta_{date_or_datetime}_stokes{stokes}.pkl')
df_mode7 = pd.read_pickle(f'{path}/df_mode7_realta_{date_or_datetime}_stokes{stokes}.pkl')

In [21]:
time_mode3 = df_mode3.index
time_mode5 = df_mode5.index
time_mode7 = df_mode7.index

freq_mode3 = df_mode3.columns
freq_mode5 = df_mode5.columns
freq_mode7 = df_mode7.columns

In [23]:
# remove the const background
mode3_new = df_mode3.values - np.tile(np.nanmean(df_mode3.values,0), (df_mode3.values.shape[0],1))
mode5_new = df_mode5.values - np.tile(np.nanmean(df_mode5.values,0), (df_mode5.values.shape[0],1))
mode7_new = df_mode7.values - np.tile(np.nanmean(df_mode7.values,0), (df_mode7.values.shape[0],1))

df_mode3_new = pd.DataFrame(data=mode3_new, columns=freq_mode3)
df_mode5_new = pd.DataFrame(data=mode5_new, columns=freq_mode5)
df_mode7_new = pd.DataFrame(data=mode7_new, columns=freq_mode7)

df_mode3_new.index = pd.to_datetime(time_mode3)
df_mode5_new.index = pd.to_datetime(time_mode5)
df_mode7_new.index = pd.to_datetime(time_mode7)

del df_mode3
del df_mode5
del df_mode7

del mode3_new
del mode5_new
del mode7_new

In [ ]:
typeII_2_mode3 = df_mode3_new.loc['2024-05-14 17:36:14':'2024-05-14 17:38:40']
typeII_2_mode5 = df_mode5_new.loc['2024-05-14 17:36:14':'2024-05-14 17:38:40']
typeII_2_mode7 = df_mode7_new.loc['2024-05-14 17:36:14':'2024-05-14 17:38:40']

min_range = -0.5
max_range = 3.7 # 2.98

fig = plt.figure(figsize=[10,7])
ax  = fig.add_subplot(111)
ax.pcolormesh(typeII_2_mode3.index, typeII_2_mode3.columns, typeII_2_mode3.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(typeII_2_mode5.index, typeII_2_mode5.columns, typeII_2_mode5.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(typeII_2_mode7.index, typeII_2_mode7.columns, typeII_2_mode7.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.set_yscale('log')

# Define the custom ticks
custom_ticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180]
ax.set_yticks(custom_ticks)
ax.set_yticklabels([str(tick) for tick in custom_ticks])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.SecondLocator(interval=5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.set_xlim(left=pd.Timestamp('2024-05-14 17:37:10'),
            right=pd.Timestamp('2024-05-14 17:37:30'))
ax.set_ylim(180, 49)
fig.savefig(f'{data_dir}/burst2_zoomin.png', format='png', bbox_inches='tight')
plt.show()

In [ ]:
typeII_1_mode3 = df_mode3_new.loc['2024-05-14 17:30:40':'2024-05-14 17:34:30']
typeII_1_mode5 = df_mode5_new.loc['2024-05-14 17:30:40':'2024-05-14 17:34:30']
typeII_1_mode7 = df_mode7_new.loc['2024-05-14 17:30:40':'2024-05-14 17:34:30']

min_range = -0.5
max_range = 3.2 # 2.98

fig = plt.figure(figsize=[12,7])
ax  = fig.add_subplot(111)
ax.pcolormesh(typeII_1_mode3.index, typeII_1_mode3.columns, typeII_1_mode3.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(typeII_1_mode5.index, typeII_1_mode5.columns, typeII_1_mode5.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(typeII_1_mode7.index, typeII_1_mode7.columns, typeII_1_mode7.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.set_yscale('log')

# Define the custom ticks
custom_ticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180]
ax.set_yticks(custom_ticks)
ax.set_yticklabels([str(tick) for tick in custom_ticks], fontsize=17)
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_xlabel('Time (UT)', fontsize=17)
ax.set_ylabel('Frequency (MHz)', fontsize=17)
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.SecondLocator(interval=5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.tick_params(axis='x', labelsize=17)
ax.set_xlim(left=pd.Timestamp('2024-05-14 17:32:00'),
            right=pd.Timestamp('2024-05-14 17:32:30'))
ax.set_ylim(190, 110)
plt.show()

## Quick tmp inspect

In [19]:
# Downsample to 1-second resolution for faster testing and visualization
df_mode3_1s = df_mode3.resample('1S').mean()
df_mode5_1s = df_mode5.resample('1S').mean()
df_mode7_1s = df_mode7.resample('1S').mean()

In [20]:
df_mode3_1s.index[0]

Timestamp('2024-05-14 10:02:00')

In [1]:
# fig = plt.figure(figsize=[12,7])
# ax = fig.add_subplot(111)
# ax.pcolormesh(df_mode3_1s.index, df_mode3_1s.columns, df_mode3_1s.values.T, cmap='RdYlBu_r') # or Spectral_r
# ax.pcolormesh(df_mode5_1s.index, df_mode5_1s.columns, df_mode5_1s.values.T, cmap='RdYlBu_r')
# ax.pcolormesh(df_mode7_1s.index, df_mode7_1s.columns, df_mode7_1s.values.T, cmap='RdYlBu_r')
# ax.set_yscale('log')
# custom_ticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]
# ax.set_yticks(custom_ticks)
# ax.set_yticklabels([str(tick) for tick in custom_ticks])
# ax.xaxis.set_minor_locator(AutoMinorLocator(n=5))
# ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
# ax.set_xlabel('Time (UT)')
# ax.set_ylabel('Frequency (MHz)')
# ax.xaxis_date()
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_ylim(ax.get_ylim()[::-1]) # reverse the y-axis
# # ax.set_xlim(left=pd.Timestamp(
# fig.tight_layout()
# plt.show()

In [23]:
# remove const bkg
df_mode3_1s_nobkg = df_mode3_1s - np.tile(np.nanmedian(df_mode3_1s,0), (df_mode3_1s.shape[0],1))
df_mode5_1s_nobkg = df_mode5_1s - np.tile(np.nanmedian(df_mode5_1s,0), (df_mode5_1s.shape[0],1))
df_mode7_1s_nobkg = df_mode7_1s - np.tile(np.nanmedian(df_mode7_1s,0), (df_mode7_1s.shape[0],1))

In [ ]:
min_range = -0.1
max_range = 1.7

fig = plt.figure(figsize=[12,7])
ax = fig.add_subplot(111)
ax.pcolormesh(df_mode3_1s_nobkg.index, df_mode3_1s_nobkg.columns, df_mode3_1s_nobkg.values.T,
              vmin=min_range, vmax=max_range,
              cmap='RdYlBu_r')
ax.pcolormesh(df_mode5_1s_nobkg.index, df_mode5_1s_nobkg.columns, df_mode5_1s_nobkg.values.T,
              vmin=min_range, vmax=max_range,
              cmap='RdYlBu_r')
ax.pcolormesh(df_mode7_1s_nobkg.index, df_mode7_1s_nobkg.columns, df_mode7_1s_nobkg.values.T,
              vmin=min_range, vmax=max_range,
              cmap='RdYlBu_r')
ax.set_yscale('log')
custom_ticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]
ax.set_yticks(custom_ticks)
ax.set_yticklabels([str(tick) for tick in custom_ticks])
ax.xaxis.set_minor_locator(AutoMinorLocator(n=5))
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_ylim(bottom=15)
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_ylim(ax.get_ylim()[::-1])
fig.tight_layout()
plt.show()

In [26]:
# Slice the DataFrame between start_date and end_date
start_date = '2024-05-14 16:34:59.999'

df_mode3_new = df_mode3.loc[start_date:]
df_mode5_new = df_mode5.loc[start_date:]
df_mode7_new = df_mode7.loc[start_date:]

In [27]:
del df_mode3
del df_mode5
del df_mode7

In [38]:
# plotting the 1s resolution
start_date = '2024-05-14 17:02:00'
end_date   = '2024-05-14 17:59:00'

mode3_1s_nobg_new = df_mode3_1s_nobkg.loc[start_date:end_date]
mode5_1s_nobg_new = df_mode5_1s_nobkg.loc[start_date:end_date]
mode7_1s_nobg_new = df_mode7_1s_nobkg.loc[start_date:end_date]

In [ ]:
min_range = -0.1
max_range = 1.7

fig = plt.figure(figsize=[12,7])
ax  = fig.add_subplot(111)
ax.pcolormesh(mode3_1s_nobg_new.index, mode3_1s_nobg_new.columns, mode3_1s_nobg_new.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(mode5_1s_nobg_new.index, mode5_1s_nobg_new.columns, mode5_1s_nobg_new.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(mode7_1s_nobg_new.index, mode7_1s_nobg_new.columns, mode7_1s_nobg_new.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.set_yscale('log')
custom_ticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200]
ax.set_yticks(custom_ticks)
ax.set_yticklabels([str(tick) for tick in custom_ticks])
ax.xaxis.set_minor_locator(AutoMinorLocator(n=5))
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_ylim(max(mode5_1s_nobg_new.columns), 20)
ax.set_ylim(max(ax.get_ylim()[::-1]), 20)
fig.show()

In [ ]:
typeII_1_mode3 = df_mode3_new.loc['2024-05-14 17:30:40':'2024-05-14 17:34:30']
typeII_1_mode5 = df_mode5_new.loc['2024-05-14 17:30:40':'2024-05-14 17:34:30']
typeII_1_mode7 = df_mode7_new.loc['2024-05-14 17:30:40':'2024-05-14 17:34:30']

typeII_2_mode3 = df_mode3_new.loc['2024-05-14 17:36:14':'2024-05-14 17:38:40']
typeII_2_mode5 = df_mode5_new.loc['2024-05-14 17:36:14':'2024-05-14 17:38:40']
typeII_2_mode7 = df_mode7_new.loc['2024-05-14 17:36:14':'2024-05-14 17:38:40']

typeII_3_mode3 = df_mode3_new.loc['2024-05-14 17:40:00':'2024-05-14 17:48:00']
typeII_3_mode5 = df_mode5_new.loc['2024-05-14 17:40:00':'2024-05-14 17:48:00']
typeII_3_mode7 = df_mode7_new.loc['2024-05-14 17:40:00':'2024-05-14 17:48:00']

typeII_4_mode3 = df_mode3_new.loc['2024-05-14 17:17:00':'2024-05-14 17:30:00']
typeII_4_mode5 = df_mode5_new.loc['2024-05-14 17:17:00':'2024-05-14 17:30:00']
typeII_4_mode7 = df_mode7_new.loc['2024-05-14 17:17:00':'2024-05-14 17:30:00']

typeII_5_mode3 = df_mode3_new.loc['2024-05-14 17:10:00':'2024-05-14 17:17:00']
typeII_5_mode5 = df_mode5_new.loc['2024-05-14 17:10:00':'2024-05-14 17:17:00']
typeII_5_mode7 = df_mode7_new.loc['2024-05-14 17:10:00':'2024-05-14 17:17:00']

typeII_6_mode3 = df_mode3_new.loc['2024-05-14 17:01:52':'2024-05-14 17:04:50']
typeII_6_mode5 = df_mode5_new.loc['2024-05-14 17:01:52':'2024-05-14 17:04:50']
typeII_6_mode7 = df_mode7_new.loc['2024-05-14 17:01:52':'2024-05-14 17:04:50']

typeII_7_mode3 = df_mode3_new.loc['2024-05-14 17:35:13':'2024-05-14 17:37:00']
typeII_7_mode5 = df_mode5_new.loc['2024-05-14 17:35:13':'2024-05-14 17:37:00']
typeII_7_mode7 = df_mode7_new.loc['2024-05-14 17:35:13':'2024-05-14 17:37:00']

CME_2_mode3 = df_mode3_new.loc['2024-05-14 17:30:20':'2024-05-14 17:48:00']
CME_2_mode5 = df_mode5_new.loc['2024-05-14 17:30:20':'2024-05-14 17:48:00']
CME_2_mode7 = df_mode7_new.loc['2024-05-14 17:30:20':'2024-05-14 17:48:00']

herring_mode3 = df_mode3_new.loc['2024-05-14 17:30:40':'2024-05-14 17:33:40']
herring_mode5 = df_mode5_new.loc['2024-05-14 17:30:40':'2024-05-14 17:33:40']
herring_mode7 = df_mode7_new.loc['2024-05-14 17:30:40':'2024-05-14 17:33:40']

In [ ]:
del df_mode3_new
del df_mode5_new
del df_mode7_new

In [ ]:
min_range = -0.5
max_range = 2.98

fig = plt.figure(figsize=[12,7])
ax = fig.add_subplot(111)
ax.pcolormesh(typeII_2_mode3.index, typeII_2_mode3.columns, typeII_2_mode3.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(typeII_2_mode5.index, typeII_2_mode5.columns, typeII_2_mode5.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.pcolormesh(typeII_2_mode7.index, typeII_2_mode7.columns, typeII_2_mode7.values.T,
              vmin=min_range, vmax=max_range, cmap='RdYlBu_r')
ax.set_yscale('log')

# Define the custom ticks
custom_ticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180]
ax.set_yticks(custom_ticks)
ax.set_yticklabels([str(tick) for tick in custom_ticks], fontsize = 17)
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_xlabel('Time (UT)', fontsize = 17)
ax.set_ylabel('Frequency (MHz)', fontsize = 17)
ax.xaxis_date()

ax.xaxis.set_major_locator(mdates.SecondLocator(interval=30))
# Define minor ticks every 2 minutes
#ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

ax.tick_params(axis='x', labelsize=17)
ax.set_ylim(180, 49)
fig.show()